<a href="https://colab.research.google.com/github/CheredlaJasvina/DSA0302-NLP/blob/main/DSA0302_AT_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 4


In [1]:

# TASK 4: NAMED ENTITY RECOGNITION

!pip install spacy
!python -m spacy download en_core_web_sm

import spacy
from collections import Counter

# Load NLP model
nlp = spacy.load("en_core_web_sm")

# Sample pension circulars
circulars = [
    "Ravi Kumar retired on 10 January 2026.",
    "Pension Directorate Chennai approved the pension.",
    "The employee receives Rs. 15000 monthly.",
    "Anitha applied for family pension on 15 March 2026."
]

# Named Entity Recognition
entity_counts = Counter()
predicted_entities = []

print("NAMED ENTITY RECOGNITION RESULTS\n")

for text in circulars:
    doc = nlp(text)

    print("Text:", text)

    for ent in doc.ents:
        print(ent.text, ":", ent.label_)
        entity_counts[ent.label_] += 1
        predicted_entities.append((ent.text, ent.label_))

# Entity count
print("\nENTITY COUNTS")

for entity, count in entity_counts.items():
    print(entity, ":", count)

# Manually annotated gold entities
# Add all 200 hand-checked mentions here.
gold_entities = [
    ("Ravi Kumar", "PERSON"),
    ("10 January 2026", "DATE"),
    ("Pension Directorate", "ORG"),
    ("Chennai", "GPE")
]

# Example evaluation on available gold annotations
correct = sum(1 for item in gold_entities
              if item in predicted_entities)

total = len(gold_entities)

accuracy = (correct / total) * 100 if total else 0

print("\nNER ACCURACY")
print("Correct Entities:", correct)
print("Total Gold Entities:", total)
print("Accuracy:", round(accuracy, 2), "%")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 95.9 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
NAMED ENTITY RECOGNITION RESULTS

Text: Ravi Kumar retired on 10 January 2026.
Ravi Kumar : PERSON
10 January 2026 : DATE
Text: Pension Directorate Chennai approved the pension.
Pension Directorate Chennai : ORG
Text: The employee receives Rs. 15000 monthly.
15000 : CARDINAL
monthly : DATE
Text: Anitha applied for family pension on 15 March 2026.
Anitha : PERSON
15 March 2026 : DATE

ENTITY COUNTS
PERSON : 2
DATE : 3
ORG : 1
CARDINAL : 1

NER ACCURACY
Correct Entities: 2
Total Gold Entities: 4
Accuracy: 50.0 %


Task 5

In [3]:

import re

def analyse(sentence):
    sentence = sentence.lower().strip().rstrip("?.")

    patterns = [
        (r"does ravi receive pension",
         "Receives(Ravi, Pension)"),

        (r"does every employee receive pension",
         "FOR ALL x: Employee(x) -> Receives(x, Pension)"),

        (r"do some employees receive pension",
         "EXISTS x: Employee(x) AND Receives(x, Pension)"),

        (r"does every dependent receive family pension",
         "FOR ALL x: Dependent(x) -> Receives(x, FamilyPension)")
    ]

    for pattern, logical_form in patterns:
        if re.fullmatch(pattern, sentence):
            return logical_form, "Success"

    if re.search(r"\b(every|some|all|any)\b", sentence):
        return None, "Quantifier or grammar not supported"

    if re.search(r"\b(receive|receives|get|gets)\b", sentence):
        return None, "Sentence structure not supported"

    return None, "Unknown word or sentence structure"


# Sample enquiries
enquiries = [
    "Does Ravi receive pension?",
    "Does every employee receive pension?",
    "Do some employees receive pension?",
    "Does every dependent receive family pension?"
]

successful = 0
failures = {}

for enquiry in enquiries:
    logical_form, status = analyse(enquiry)

    if status == "Success":
        successful += 1
        print(enquiry, "->", logical_form)
    else:
        failures[status] = failures.get(status, 0) + 1
        print(enquiry, "->", status)

total = len(enquiries)
coverage = (successful / total) * 100

print("\nTotal enquiries:", total)
print("Successful analyses:", successful)
print("Coverage:", coverage, "%")
print("Failures:", failures)


Does Ravi receive pension? -> Receives(Ravi, Pension)
Does every employee receive pension? -> FOR ALL x: Employee(x) -> Receives(x, Pension)
Do some employees receive pension? -> EXISTS x: Employee(x) AND Receives(x, Pension)
Does every dependent receive family pension? -> FOR ALL x: Dependent(x) -> Receives(x, FamilyPension)

Total enquiries: 4
Successful analyses: 4
Coverage: 100.0 %
Failures: {}


Task7

In [4]:

import re
from collections import Counter, defaultdict

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.naive_bayes import MultinomialNB

# Possible senses and their glosses
glosses = {
    "service": {
        "employment": "employee work job occupation years completed",
        "assistance": "help support aid customer service provided"
    },
    "award": {
        "benefit": "pension benefit financial grant payment",
        "prize": "achievement winner medal prize recognition"
    },
    "dependent": {
        "family": "child family member relative supported employee",
        "conditional": "depends relies condition factor dependent"
    }
}

# Related gloss words for extended Lesk
related = {
    "service": {
        "employment": "career salary retirement staff",
        "assistance": "helpdesk support guidance information"
    },
    "award": {
        "benefit": "allowance entitlement pension scheme",
        "prize": "honour trophy reward achievement"
    },
    "dependent": {
        "family": "spouse child parent beneficiary",
        "conditional": "dependent upon subject to requirement"
    }
}

stop_words = {
    "the", "is", "a", "an", "and", "of", "to",
    "his", "her", "for", "in", "on", "was"
}

def tokenize(text, remove_stop=False):
    words = re.findall(r"\b[a-z]+\b", text.lower())
    if remove_stop:
        words = [w for w in words if w not in stop_words]
    return set(words)

def lesk(word, context, remove_stop=False, extended=False):
    context_words = tokenize(context, remove_stop)
    scores = {}

    for sense, gloss in glosses[word].items():
        signature = gloss

        if extended:
            signature += " " + related[word][sense]

        signature_words = tokenize(signature, remove_stop)
        scores[sense] = len(context_words & signature_words)

    selected = max(scores, key=scores.get)
    return selected, scores


# Test enquiries
queries = [
    ("service", "The employee completed years of work."),
    ("award", "The officer approved the pension benefit."),
    ("dependent", "The employee added his child to the application.")
]

print("SIMPLIFIED LESK RESULTS")

for word, sentence in queries:
    sense, scores = lesk(word, sentence)
    print("\nWord:", word)
    print("Sentence:", sentence)
    print("Overlap scores:", scores)
    print("Selected sense:", sense)


# Compare Lesk variations
print("\nLESK METHOD COMPARISON")

for method, remove, extend in [
    ("Basic Lesk", False, False),
    ("Stop-word removal", True, False),
    ("Extended Lesk", False, True),
    ("Extended + stop-word removal", True, True)
]:
    print("\n", method)

    for word, sentence in queries:
        sense, scores = lesk(
            word, sentence,
            remove_stop=remove,
            extended=extend
        )
        print(word, "->", sense, scores)


# Annotated sample dataset
# Replace these examples with your actual annotated data.
data = []

examples = {
    ("service", "employment"): [
        "employee completed years of work",
        "government employee finished service period",
        "staff retirement after long employment",
        "employee salary during working career",
        "years of employment in department"
    ],
    ("service", "assistance"): [
        "helpdesk provides support to employee",
        "customer received assistance from office",
        "staff offered help and guidance",
        "employee requested information from helpdesk",
        "support service answered the question"
    ],
    ("award", "benefit"): [
        "pension benefit granted to employee",
        "officer approved financial payment",
        "retirement allowance awarded to staff",
        "employee received pension entitlement",
        "family pension benefit was granted"
    ],
    ("award", "prize"): [
        "winner received a medal for achievement",
        "prize given for excellent performance",
        "student won recognition and trophy",
        "award ceremony honoured the winner",
        "achievement received a special prize"
    ],
    ("dependent", "family"): [
        "employee added child to family pension",
        "spouse is eligible for pension benefit",
        "dependent family member receives support",
        "employee supports his child",
        "parent is listed as a beneficiary"
    ],
    ("dependent", "conditional"): [
        "result depends on another condition",
        "payment relies on eligibility factors",
        "decision is subject to requirements",
        "outcome depends on the given condition",
        "approval relies on specified criteria"
    ]
}

for (word, sense), sentences in examples.items():
    for sentence in sentences:
        data.append((word, sentence, sense))

# Most-Frequent-Sense baseline
counts = defaultdict(Counter)

for word, sentence, sense in data:
    counts[word][sense] += 1

mfs = {
    word: counter.most_common(1)[0][0]
    for word, counter in counts.items()
}

correct = 0

for word, sentence, actual in data:
    if mfs[word] == actual:
        correct += 1

print("\nMOST FREQUENT SENSE BASELINE")
print("Accuracy:", round(correct / len(data) * 100, 2), "%")


# Supervised classifier using contextual text features
X = [word + " " + sentence for word, sentence, sense in data]
y = [word + "_" + sense for word, sentence, sense in data]

model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2)),
    MultinomialNB()
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

print("\nFIVE-FOLD CLASSIFICATION")
print("Fold accuracies:", scores)
print("Overall accuracy:", round(scores.mean() * 100, 2), "%")

# Per-term accuracy using five-fold predictions
from sklearn.model_selection import cross_val_predict

predictions = cross_val_predict(model, X, y, cv=cv)

print("\nPER-TERM ACCURACY")

for word in glosses:
    indices = [
        i for i, item in enumerate(data)
        if item[0] == word
    ]

    term_correct = sum(
        predictions[i] == y[i] for i in indices
    )

    accuracy = term_correct / len(indices) * 100
    print(word, ":", round(accuracy, 2), "%")


SIMPLIFIED LESK RESULTS

Word: service
Sentence: The employee completed years of work.
Overlap scores: {'employment': 4, 'assistance': 0}
Selected sense: employment

Word: award
Sentence: The officer approved the pension benefit.
Overlap scores: {'benefit': 2, 'prize': 0}
Selected sense: benefit

Word: dependent
Sentence: The employee added his child to the application.
Overlap scores: {'family': 2, 'conditional': 0}
Selected sense: family

LESK METHOD COMPARISON

 Basic Lesk
service -> employment {'employment': 4, 'assistance': 0}
award -> benefit {'benefit': 2, 'prize': 0}
dependent -> family {'family': 2, 'conditional': 0}

 Stop-word removal
service -> employment {'employment': 4, 'assistance': 0}
award -> benefit {'benefit': 2, 'prize': 0}
dependent -> family {'family': 2, 'conditional': 0}

 Extended Lesk
service -> employment {'employment': 4, 'assistance': 0}
award -> benefit {'benefit': 2, 'prize': 0}
dependent -> family {'family': 2, 'conditional': 1}

 Extended + stop-word r

Task8

In [5]:

import re
import math
from collections import defaultdict

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Sample pension circulars
documents = {
    "D1": "Employee pension eligibility and retirement benefits",
    "D2": "Family pension benefits for dependent members",
    "D3": "Employee retirement age and service rules",
    "D4": "Dependent family members pension eligibility",
    "D5": "Retirement service rules for government employees",
    "D6": "Pension payment benefits for retired employees"
}

# Six search queries
queries = {
    "Q1": "Employee pension eligibility",
    "Q2": "Family pension benefits",
    "Q3": "Employee retirement age",
    "Q4": "Dependent family pension",
    "Q5": "Retirement service rules",
    "Q6": "Pension benefits for employees"
}

# Relevant documents for each query (sample judgments)
relevant = {
    "Q1": {"D1"},
    "Q2": {"D2"},
    "Q3": {"D3"},
    "Q4": {"D4"},
    "Q5": {"D5"},
    "Q6": {"D6"}
}

# Tokenization, stop-word removal and stemming
# Simple stemming is demonstrated using word endings.
def preprocess(text):
    words = re.findall(r"\b[a-z]+\b", text.lower())

    stop_words = {
        "the", "is", "a", "an", "and", "of",
        "to", "for", "in", "on", "with"
    }

    result = []

    for word in words:
        if word not in stop_words:
            if word.endswith("ies"):
                word = word[:-3] + "y"
            elif word.endswith("s") and len(word) > 3:
                word = word[:-1]

            result.append(word)

    return " ".join(result)


doc_ids = list(documents.keys())
doc_texts = [preprocess(documents[d]) for d in doc_ids]

# Build TF-IDF matrix
vectorizer = TfidfVectorizer()
doc_matrix = vectorizer.fit_transform(doc_texts)

print("TERM-DOCUMENT MATRIX")
print(vectorizer.get_feature_names_out())
print(doc_matrix.toarray().round(3))

# Build inverted index
inverted_index = defaultdict(list)

for doc_id, text in zip(doc_ids, doc_texts):
    for term in set(text.split()):
        inverted_index[term].append(doc_id)

print("\nINVERTED INDEX")
for term, ids in sorted(inverted_index.items()):
    print(term, ":", ids)


# Rank documents using cosine similarity
def retrieve(query):
    query_vector = vectorizer.transform([preprocess(query)])
    scores = cosine_similarity(query_vector, doc_matrix)[0]

    ranking = sorted(
        zip(doc_ids, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return ranking


print("\nLEXICAL RETRIEVAL RESULTS")

lexical_results = {}

for qid, query in queries.items():
    ranking = retrieve(query)
    lexical_results[qid] = [doc for doc, score in ranking]

    print("\n", qid, ":", query)
    for doc, score in ranking:
        print(doc, "Similarity:", round(score, 3))


# Evaluation metrics
def evaluate(results, relevant_docs):
    relevant_retrieved = [
        doc for doc in results if doc in relevant_docs
    ]

    tp = len(relevant_retrieved)
    fp = len(results) - tp
    fn = len(relevant_docs) - tp

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    p_at_10 = sum(
        1 for doc in results[:10] if doc in relevant_docs
    ) / 10

    # Average Precision
    hits = 0
    total_precision = 0

    for rank, doc in enumerate(results, start=1):
        if doc in relevant_docs:
            hits += 1
            total_precision += hits / rank

    ap = (
        total_precision / len(relevant_docs)
        if relevant_docs else 0
    )

    return precision, recall, f1, p_at_10, ap


def print_evaluation(result_set):
    totals = [0, 0, 0, 0, 0]

    print("\nQuery | Precision | Recall | F1 | P@10 | AP")

    for qid in queries:
        values = evaluate(
            result_set[qid],
            relevant[qid]
        )

        totals = [
            a + b for a, b in zip(totals, values)
        ]

        print(qid, *[round(v, 3) for v in values])

    print("\nMean scores:")
    print([
        round(value / len(queries), 3)
        for value in totals
    ])


print("\nLEXICAL RETRIEVAL EVALUATION")
print_evaluation(lexical_results)


# Sense-aware query expansion
expansion = {
    "service": "employment work assistance",
    "award": "pension benefit grant",
    "dependent": "family child member",
    "pension": "retirement payment allowance"
}

def expand_query(query):
    words = query.lower().split()
    extra_words = []

    for word in words:
        if word in expansion:
            extra_words.extend(expansion[word].split())

    return query + " " + " ".join(extra_words)


expanded_results = {}

for qid, query in queries.items():
    expanded_query = expand_query(query)
    ranking = retrieve(expanded_query)
    expanded_results[qid] = [doc for doc, score in ranking]

print("\nSENSE-AWARE RETRIEVAL EVALUATION")
print_evaluation(expanded_results)


TERM-DOCUMENT MATRIX
['age' 'benefit' 'dependent' 'eligibility' 'employee' 'family'
 'government' 'member' 'payment' 'pension' 'retired' 'retirement' 'rule'
 'service']
[[0.    0.453 0.    0.537 0.388 0.    0.    0.    0.    0.388 0.    0.453
  0.    0.   ]
 [0.    0.41  0.486 0.    0.    0.486 0.    0.486 0.    0.352 0.    0.
  0.    0.   ]
 [0.561 0.    0.    0.    0.333 0.    0.    0.    0.    0.    0.    0.388
  0.46  0.46 ]
 [0.    0.    0.47  0.47  0.    0.47  0.    0.47  0.    0.34  0.    0.
  0.    0.   ]
 [0.    0.    0.    0.    0.333 0.    0.561 0.    0.    0.    0.    0.388
  0.46  0.46 ]
 [0.    0.388 0.    0.    0.333 0.    0.    0.    0.56  0.333 0.56  0.
  0.    0.   ]]

INVERTED INDEX
age : ['D3']
benefit : ['D1', 'D2', 'D6']
dependent : ['D2', 'D4']
eligibility : ['D1', 'D4']
employee : ['D1', 'D3', 'D5', 'D6']
family : ['D2', 'D4']
government : ['D5']
member : ['D2', 'D4']
payment : ['D6']
pension : ['D1', 'D2', 'D4', 'D6']
retired : ['D6']
retirement : ['D1', 'D3', 

Task 9

In [6]:

from sklearn.metrics import cohen_kappa_score

def calculate_agreement(annotator1, annotator2, task_name):
    total = len(annotator1)

    agreements = sum(
        a == b for a, b in zip(annotator1, annotator2)
    )

    raw_agreement = (agreements / total) * 100
    kappa = cohen_kappa_score(annotator1, annotator2)

    print("\n", task_name)
    print("Total annotations:", total)
    print("Agreements:", agreements)
    print("Raw agreement:", round(raw_agreement, 2), "%")
    print("Cohen's Kappa:", round(kappa, 3))


# DEMONSTRATION DATA ONLY
# Replace with actual independent annotations.

# Sense labels for 10 sample items
sense_annotator1 = [
    "employment", "assistance", "family", "benefit", "employment",
    "family", "prize", "assistance", "benefit", "employment"
]

sense_annotator2 = [
    "employment", "assistance", "family", "benefit", "assistance",
    "family", "prize", "assistance", "benefit", "employment"
]

# Relevance labels for 15 sample queries
relevance_annotator1 = [
    1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0
]

relevance_annotator2 = [
    1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0
]

calculate_agreement(
    sense_annotator1,
    sense_annotator2,
    "Sense Annotation"
)

calculate_agreement(
    relevance_annotator1,
    relevance_annotator2,
    "Relevance Judgement"
)



 Sense Annotation
Total annotations: 10
Agreements: 9
Raw agreement: 90.0 %
Cohen's Kappa: 0.873

 Relevance Judgement
Total annotations: 15
Agreements: 14
Raw agreement: 93.33 %
Cohen's Kappa: 0.865


Task 10

In [8]:

def lambda_reduce(expression):
    match = re.fullmatch(
        r"\(lambda\s+(\w+)\s+(.+)\)\s+(\w+)",
        expression.strip()
    )
    if match:
        variable, body, argument = match.groups()
        return re.sub(
            rf"\b{variable}\b",
            argument,
            body
        )
    return expression

# Test cases
assert lambda_reduce(
    "(lambda x receive(x,pension)) Ravi"
) == "receive(Ravi,pension)"

assert pension_eligible("Ravi") == True
assert pension_eligible("Arun") == False
assert len(wsd_results) == len(queries)
assert rankings.shape == (len(queries), len(circulars))

print("Lambda Reduction:", lambda_reduce(
    "(lambda x receive(x,pension)) Ravi"
))
print("Ravi Eligible:", pension_eligible("Ravi"))
print("Arun Eligible:", pension_eligible("Arun"))
print("All test cases passed.")


Lambda Reduction: receive(Ravi,pension)
Ravi Eligible: True
Arun Eligible: False
All test cases passed.
